# 02 Automatic Vectorization

ML models process batches of data, not individual examples. Manually rewriting single-example code for batches is tedious and error-prone. `jax.vmap()` lets you write code for a single example and automatically lifts it to work on entire batches — cleanly and efficiently.

Original Documentation: https://docs.jax.dev/en/latest/automatic-vectorization.html

In [16]:
import jax
import jax.numpy as jnp

## Manual vectorization


In [17]:
def convolve(x, w):
    output = []
    for i in range(1, len(x) - 1):
        output.append(jnp.dot(x[i - 1 : i + 2], w))
    return jnp.array(output)


x = jnp.arange(5)
w = jnp.array([2.0, 3.0, 4.0])

print(convolve(x, w))

[11. 20. 29.]


Suppose we wanted to apply `convolve` to a batch of weights `w` and vectors `x`. The naive implementation would introduce a new Python loop that calls `convolve`:


In [18]:
def convolve(x, w):
    output = []
    for i in range(1, len(x) - 1):
        output.append(jnp.dot(x[i - 1 : i + 2], w))
    return jnp.array(output)


x = jnp.arange(5)
w = jnp.array([2.0, 3.0, 4.0])

# Sample batch of vectors and weights
xs = jnp.stack([x, x])
ws = jnp.stack([w, w])


def naive_batch_convolve(xs, ws):
    assert xs.shape[0] == ws.shape[0]
    output = []
    for i in range(xs.shape[0]):
        output.append(convolve(xs[i], ws[i]))
    return jnp.stack(output)


print(naive_batch_convolve(xs, ws))

[[11. 20. 29.]
 [11. 20. 29.]]


This is quite inefficient since each batch’s convolution could have been done in parallel. To make it more efficient, we could rewrite to express the computations in a batched form:


In [19]:
def manual_batch_convolve(xs, ws):
    assert xs.shape[0] == ws.shape[0]
    output = []
    for i in range(1, xs.shape[1] - 1):  # xs.shape[-1] == 5
        # Apply convolution along axis=1
        # xs[:, i - 1 : i + 2] is sliding window of size 3 across axis=1
        output.append(jnp.sum(xs[:, i - 1 : i + 2] * ws, axis=1))
    return jnp.stack(output, axis=1)


print(manual_batch_convolve(xs, ws))

[[11. 20. 29.]
 [11. 20. 29.]]


Here, we compute the convolution for all windows in the batch simultaneously at each iteration by sliding a window along the last axis. In other words, each loop iteration processes the same window across all batches in parallel (since we use `jnp.sum()`)

However, this is messy and error-prone.

## Automatic vectorization

`jax.vmap()` automatically generates a vectorized implementation of a function:


In [20]:
auto_batch_convolve = jax.vmap(convolve)

print(auto_batch_convolve(xs, ws))

[[11. 20. 29.]
 [11. 20. 29.]]


If the batch dimension is not the first axis, use the `in_axes` and `out_axes` arguments to specify the location of the batch dimension in inputs and outputs.


In [21]:
# Transpose so batch dimension is now axis=1
xst = xs.transpose()  # shape (5, 2)
wst = ws.transpose()  # shape (3, 2)

auto_batch_convolve_v2 = jax.vmap(convolve, in_axes=1, out_axes=1)

print(auto_batch_convolve_v2(xst, wst))

[[11. 11.]
 [20. 20.]
 [29. 29.]]


## Combining transformations

`jax.jit()` and `jax.vmap()` are composable; we can wrap them in either order.

Notably, the order of wrapping them does matter: JIT compiling a vectorized function is different from vectorizing a JIT compiled function.

Usually, you want to JIT compile a vectorized function (`jax.jit(jax.vmap(fn))`) since it gives the XLA compiler more code to optmize upfront.

However, if the shapes of batches are not consistent, then we want to vectorize a JIT compiled function (`jax.vmap(jax.jit(fn))`) since we cannot provide different shapes to a function that has been compiled for one shape.


In [22]:
compiled_batch_convolve = jax.jit(
    jax.vmap(convolve)
)  # Shape along axis=1 is consistent

print(compiled_batch_convolve(xs, ws))

[[11. 20. 29.]
 [11. 20. 29.]]


## Performance comparison

Let's compare the performance of each approach with larger inputs:

In [ ]:
import time

# Larger inputs for meaningful timing
key = jax.random.key(0)
xs_large = jax.random.normal(key, (1000, 50))
ws_large = jax.random.normal(key, (1000, 3))

def convolve(x, w):
    output = []
    for i in range(1, len(x) - 1):
        output.append(jnp.dot(x[i - 1 : i + 2], w))
    return jnp.array(output)

# Naive batch (Python loop)
def naive_batch(xs, ws):
    return jnp.stack([convolve(xs[i], ws[i]) for i in range(xs.shape[0])])

# vmap (auto-vectorized)
vmap_batch = jax.jit(jax.vmap(convolve))

# Warm up JIT
vmap_batch(xs_large, ws_large).block_until_ready()

start = time.time()
naive_batch(xs_large, ws_large).block_until_ready()
naive_time = time.time() - start

start = time.time()
vmap_batch(xs_large, ws_large).block_until_ready()
vmap_time = time.time() - start

print(f"Naive loop:  {naive_time:.4f}s")
print(f"jit(vmap):   {vmap_time:.4f}s")
print(f"Speedup:     {naive_time / vmap_time:.0f}x")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 2.5))
bars = ax.barh(
    ["Naive Python loop", "jit(vmap)"],
    [naive_time * 1000, vmap_time * 1000],
)
ax.set_xlabel("Time (ms)")
ax.set_title("Batched Convolution: Loop vs vmap")
ax.bar_label(bars, fmt="%.1f ms", padding=3)
plt.tight_layout()
plt.show()

---

## Summary

| Function | Purpose |
|---|---|
| `jax.vmap(f)` | Automatically vectorize `f` to work on batches |
| `jax.vmap(f, in_axes=..., out_axes=...)` | Specify which axis is the batch dimension for each input/output |
| `jax.jit(jax.vmap(f))` | Compile the vectorized function for maximum performance |

**Key takeaways:**
- `vmap` transforms single-example code into batched code automatically
- Use `in_axes` and `out_axes` when the batch dimension isn't axis 0
- `jit(vmap(f))` is usually the best composition order — gives XLA more to optimize
- Use `vmap(jit(f))` only when batch shapes vary

## Exercises

**Exercise 1:** Write a function `euclidean_distance(a, b)` that computes the Euclidean distance between two vectors. Use `vmap` to compute pairwise distances between all pairs in two batches of points. Verify against a manual loop.

In [ ]:
# Your code here

**Exercise 2:** Given a batch of matrices and a batch of vectors, use `vmap` to compute the matrix-vector product for each pair. Experiment with `in_axes` to handle the case where you have one matrix but many vectors.

In [ ]:
# Your code here

**Exercise 3:** Use `vmap` to compute the Jacobian of a function $f: \mathbb{R}^n \rightarrow \mathbb{R}^m$ by vmapping `jax.grad` over the output dimensions. Compare with `jax.jacobian`.

In [ ]:
# Your code here